In [ ]:
!pip install newsapi-python

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
from newsapi import NewsApiClient

newsapi = NewsApiClient(api_key='')
dtformat = '%Y-%m-%d'
end_time = datetime(2020, 12, 31)
start_time = datetime(2019, 12, 1)
df = pd.DataFrame()
day = 1

whitespace = re.compile(r"\s+")
num = re.compile(r"[0-9]")
mention = re.compile(r"@\S+")
hashtag = re.compile(r"#\s+")
word_in_bracket = re.compile(r"(\(.*\))|(\[.*\])")
web_address = re.compile(r"(?i)https(s):\/\/[a-z0-9.~_\-\/]+")
special_char = re.compile(r"[^a-zA-Z0-9\s]+")

def go_back(now, day):
    return now - timedelta(days=day)

def clean(news):
  news = whitespace.sub(' ', news)
  news = num.sub('', news)
  news = mention.sub('', news)
  news = hashtag.sub('', news)
  news = word_in_bracket.sub('', news)
  news = web_address.sub('', news)
  news = special_char.sub('', news)
  return news

In [ ]:
while True:

    if end_time <= start_time:
        break
    pre1 = go_back(end_time, day)
    start_time_input = pre1.strftime(dtformat)
    end_time_input = end_time.strftime(dtformat)
    print(start_time_input)
    print(end_time_input)

    articles = newsapi.get_everything(q='Hong Kong',
                                      sources='cbs-news,reuters,cnn,the-washington-post,the-washington-times,cbs-news,associated-press',
                                      from_param=start_time_input,
                                      to=end_time_input,
                                      language='en',
                                      sort_by='relevancy',
                                      page=1)
    
    for news in articles['articles']:
        #print(news)
        row = {'created_at': news['publishedAt'][0:10],
               'title': news['title'],
               'description': clean(str(news['description'])),
               'content': clean(str(news['content'])),
               'url': news['url'],
               'source': news['source']['id']}
        df = df.append(row, ignore_index=True)

    end_time = pre1

print(df)
df.to_csv('./news_2020.csv', index=False, encoding='utf-8')